# AP02 Spick – In a nutshell
### Data Preparation · Data Acquisition · EDA · Mock Exam

Ein kompakter Nachschlage-Spick aus **beiden Original-Notebooks** und dem **AP02 Mock Exam**, mit den unterschiedlichen Methoden und wichtigen Varianten, aber **ohne lange Übungspipelines**

- **Eine Prüfungsaktion = ein englischer Kommentar + möglichst genau eine Python-Codezeile**
- **ANPASSEN:** `df` (dein DataFrame), `df2` (zweiter DataFrame), Spaltennamen, Grenzwerte, Pfade und Labels. `df_raw`/`df_clean` entsprechen den Namen aus dem Mock Exam; hier arbeiten die Muster einheitlich mit `df`
- **Die grosse Codezelle ist eine Sammlung von Alternativen, kein ausführbares Gesamtskript.** Einzelne Befehle passend zur Aufgabe kopieren. Insbesondere nicht alle `dropna`-, `drop_duplicates`-, Encoding- oder Transformationsvarianten nacheinander anwenden
- **Prüfungspriorität:** ★ im Kommentar kennzeichnet direkt im Mock Exam verwendete oder zur Fehleranalyse notwendige Befehle. Andere Methoden aus den Originalspicks bleiben ebenfalls enthalten
- Voraussetzungen: Bei DataFrames mit unpassenden Spaltennamen zuerst `df.columns.tolist()` ansehen; numerische Verfahren verlangen numerische Spalten. Für `sklearn`-Scaler jeweils nur die entsprechende Zeile auswählen; bei ML Train-/Test-Split den Scaler nur auf Trainingsdaten fitten

**Inhaltsübersicht der einen Codezelle:** Setup & Import · Datenüberblick · Auswahl & Sortierung · Missing Values · Duplikate · Regex & Text · Neue Variablen & Datum · Ausreisser/EDA · Discretization · One-Hot Encoding · Scaling · Data Transformation · Combining/Reshaping/Aggregation · Korrelation & Grafiken · HTML/Web Scraping


## 01. Libraries, Paths & Importing Data

In [ ]:
# ===== 01. LIBRARIES, PATHS & IMPORTING DATA =====

# Import pandas ★
import pandas as pd

# Import NumPy
import numpy as np

# Import regular expressions ★
import re

# Import Matplotlib ★
import matplotlib.pyplot as plt

# Import Seaborn ★
import seaborn as sns

# Import BeautifulSoup for HTML parsing ★
from bs4 import BeautifulSoup

# Import Path for local HTML files
from pathlib import Path

# Import datetime
from datetime import datetime

# Import scikit-learn preprocessing methods
from sklearn.preprocessing import KBinsDiscretizer, OneHotEncoder, MinMaxScaler, MaxAbsScaler, RobustScaler, StandardScaler

# Import Box-Cox transformation
from scipy.stats import boxcox

# Import SciPy statistics (Q-Q plot)
from scipy import stats

# Print the current working directory
print(Path.cwd())

# List files in the current working directory
print([p.name for p in Path.cwd().iterdir()])

# Read a comma-separated CSV file ★
df = pd.read_csv('data.csv')

# Load a CSV with custom missing-value markers
df = pd.read_csv('data.csv', na_values=['-', 'n/a', 'unknown'])

# Read a semicolon-separated CSV file
df = pd.read_csv('data.csv', sep=';', encoding='utf-8')

# Read a CSV and preserve leading zeros in an ID field
df = pd.read_csv('data.csv', dtype={'product_id': 'string'})

# Read a JSON file ★
df2 = pd.read_json('reviews.json')

# Read an HTML file as text ★
html_content = Path('html/product_detail.html').read_text(encoding='utf-8')

# Create an independent working copy (do not modify the source) ★
df = df_raw.copy()

## 02. Inspect Dataframes & Data Types

In [ ]:
# ===== 02. INSPECT DATAFRAMES & DATA TYPES =====

# Show the first five rows ★
df.head()

# Show the last five rows
df.tail()

# Show the number of rows and columns ★
df.shape

# Count rows ★
df.shape[0]

# Count columns
df.shape[1]

# Show all column names
df.columns.tolist()

# Show data types
df.dtypes

# Show columns, data types and non-null counts ★
df.info()

# Show descriptive statistics ★
df.describe().round(2)

# Show descriptive statistics for all columns
df.describe(include='all')

# Describe one numerical variable
df['price_chf'].describe().round(2)

# Describe one categorical variable
df['brand'].describe()

# Describe a numeric variable separately for each category
df.groupby('brand')['price_chf'].describe().round(2)

# Count non-missing values per column
df.count()

# Count unique values in a column
df['brand'].nunique(dropna=True)

# Show unique values
df['brand'].unique()

# Count occurrences per category including missing values
df['brand'].value_counts(dropna=False)

# Show relative category frequencies in percent
df['brand'].value_counts(normalize=True, dropna=False).mul(100).round(2)

# Calculate cumulative counts per category (ordered by frequency)
df['brand'].value_counts().cumsum()

# Display the ten most frequent categories
df['brand'].value_counts().head(10)

# Select columns of a numeric data type
df.select_dtypes(include='number')

# Show random rows for a quick data check
df.sample(n=5, random_state=42)

# Convert a column to text (preserve missing values)
df['product_id'] = df['product_id'].astype('string')

# Convert a numeric column to decimal numbers
df['price_chf'] = pd.to_numeric(df['price_chf'], errors='coerce').astype('float64')

# Convert a text variable to a category
df['brand_category'] = df['brand'].astype('category')

# Convert an exact text match to a nullable boolean
df['available_bool'] = df['availability'].astype('string').eq('Sofort lieferbar').astype('boolean')

# Convert nullable booleans to 0/1, preserving missing values
df['available_binary'] = df['available_bool'].astype('Int64')

# Convert column names to lowercase
df.columns = df.columns.str.lower()

# Convert a column to numeric values (invalid text becomes NaN) ★
df['price_chf'] = pd.to_numeric(df['price_raw'], errors='coerce')

# Convert numeric values to nullable integers
df['count'] = pd.to_numeric(df['count'], errors='coerce').astype('Int64')

# Reset the index after removing rows ★
df = df.reset_index(drop=True)

## 03. Select, Filter, Sort & Compare

In [ ]:
# ===== 03. SELECT, FILTER, SORT & COMPARE =====

# Select one column
df['price_chf']

# Select multiple columns ★
df = df[['product_id', 'brand', 'price_chf']].copy()

# Select rows by condition ★
df_filtered = df.loc[df['price_chf'] > 1000].copy()

# Select rows between two limits
df_filtered = df.loc[df['price_chf'].between(500, 1000)].copy()

# Select records from selected brands
df_filtered = df.loc[df['brand'].isin(['Apple', 'Samsung'])].copy()

# Combine conditions with AND
df_filtered = df.loc[(df['brand'] == 'Samsung') & (df['price_chf'] > 1000)].copy()

# Combine conditions with OR
df_filtered = df.loc[(df['brand'] == 'Samsung') | (df['price_chf'] > 1000)].copy()

# Select the first five rows with iloc
df.iloc[:5]

# Filter selected columns with loc
df.loc[df['price_chf'] > 1000, ['brand', 'price_chf']]

# Sort prices from highest to lowest ★
df.sort_values('price_chf', ascending=False).head(10)

# Sort by two columns (different directions)
df.sort_values(['price_chf', 'mileage'], ascending=[False, True])

# Display columns containing a particular text
df.filter(like='price_category_').head()

# Select rows with an EXACT text value
df.loc[df['brand'] == 'Samsung']

# Exclude rows with an exact text value
df.loc[df['brand'] != 'Samsung']

# Find text anywhere in a column (case-insensitive, literal match)
df.loc[df['model'].astype('string').str.contains('Pro', case=False, na=False, regex=False)]

# Find text matching a regex pattern
df.loc[df['model'].astype('string').str.contains(r'Pro|Ultra', case=False, na=False, regex=True)]

# Find values beginning with a prefix
df.loc[df['model'].astype('string').str.startswith('Galaxy', na=False)]

# Find values ending with a suffix
df.loc[df['model'].astype('string').str.endswith('Pro', na=False)]

# Search for text across ALL columns
df.loc[df.astype('string').apply(lambda s: s.str.contains('Anfrage', case=False, na=False, regex=False)).any(axis=1)]

# Keep rows with an exact numeric value
df.loc[df['price_chf'] == 999]

# Exclude rows with an exact numeric value
df.loc[df['price_chf'] != 999]

# Keep rows where a value is greater than or equal to a threshold
df.loc[df['price_chf'] >= 1000]

# Keep rows where a value is less than or equal to a threshold
df.loc[df['price_chf'] <= 500]

# Find one of several numeric values
df.loc[df['price_chf'].isin([499, 799, 999])]

# Negate a condition (note: missing-value behavior depends on dtype)
df.loc[~df['price_chf'].between(500, 1000)]

# Keep rows with valid prices
df.loc[df['price_chf'].notna()]

# Combine a list of categories with a numerical interval
df.loc[df['brand'].isin(['Apple', 'Samsung']) & df['price_chf'].between(500, 1000)]

# Sort ascending (smallest values first)
df.sort_values('price_chf', ascending=True).head(10)

# Find the ten largest numeric values with their rows
df.nlargest(10, 'price_chf')

# Find the ten smallest numeric values with their rows
df.nsmallest(10, 'price_chf')

# Find the smallest and largest numerical values
df['price_chf'].agg(['min', 'max'])

# Reorder or restrict columns by an explicit list
df.reindex(columns=['product_id', 'brand', 'price_chf'])

# Map category labels to numerical codes or new strings
df['brand_code'] = df['brand'].map({'Apple': 1, 'Samsung': 2})

# Check whether a scalar is missing
pd.isna(np.nan)

# Check whether all entries in selected columns are valid
df[['brand', 'price_chf']].notna().all(axis=1).sum()

# Check whether a column has a numeric dtype
pd.api.types.is_numeric_dtype(df['price_chf'])

## 04. Missing Values, Imputation

In [ ]:
# ===== 04. MISSING VALUES / IMPUTATION =====

# Count missing values per column ★
df.isna().sum()

# Count all missing cells ★
df.isna().sum().sum()

# Count missing values in a specific column ★
df['price_chf'].isna().sum()

# Calculate the percentage of missing values in a column
df['price_chf'].isna().mean() * 100

# Count rows containing at least one missing value
df.isna().any(axis=1).sum()

# Show rows containing any missing values
df.loc[df.isna().any(axis=1)]

# Show rows with a missing price
df.loc[df['price_chf'].isna()]

# Show missing-value rate by brand (size=count, sum=missing, mean=rate)
df['price_chf'].isna().groupby(df['brand'], dropna=False).agg(['size', 'sum', 'mean'])

# Drop all rows containing at least one missing value
df = df.dropna()

# Drop rows with missing values in selected columns
df = df.dropna(subset=['brand', 'price_chf'])

# Drop rows only if ALL selected columns are missing
df = df.dropna(subset=['brand', 'price_chf'], how='all')

# Drop rows only if ALL columns are missing
df = df.dropna(how='all')

# Drop columns containing at least one missing value
df = df.dropna(axis=1)

# Drop columns only if ALL values are missing
df = df.dropna(axis=1, how='all')

# Fill missing prices with the mean (new column)
df['price_mean_filled'] = df['price_chf'].fillna(df['price_chf'].mean())

# Fill missing prices with the median (new column)
df['price_median_filled'] = df['price_chf'].fillna(df['price_chf'].median())

# Fill missing text with a fixed value
df['brand_filled'] = df['brand'].fillna('Unknown')

# Confirm that no prices are missing after imputation
df['price_median_filled'].isna().sum()

# Replace missing prices with a median computed within each category
df['price_brand_imputed'] = df['price_chf'].fillna(df.groupby('brand')['price_chf'].transform('median'))

# Count missing values using isnull() (alias for isna())
df['price_chf'].isnull().sum()

## 05. Duplicated Values & Index Alignment

In [ ]:
# ===== 05. DUPLICATED VALUES & INDEX ALIGNMENT =====

# Count fully duplicated rows ★
df.duplicated().sum()

# Show duplicated rows (except the first occurrence)
df.loc[df.duplicated()]

# Show ALL occurrences of duplicated rows
df.loc[df.duplicated(keep=False)]

# Count duplicates based on a unique key
df.duplicated(subset=['product_id']).sum()

# Show all records with duplicate product IDs
df.loc[df.duplicated(subset=['product_id'], keep=False)]

# Drop rows with duplicated values (if any) ★
df = df.drop_duplicates()

# Drop duplicates by key (keep the first record)
df = df.drop_duplicates(subset=['product_id'], keep='first')

# Drop duplicates by key (keep the last record)
df = df.drop_duplicates(subset=['product_id'], keep='last')

# Drop ALL rows in a duplicate group
df = df.drop_duplicates(subset=['product_id'], keep=False)

# Drop duplicates and reset the index in one line ★
df = df.drop_duplicates().reset_index(drop=True)

# Check duplicates after cleaning
df.duplicated().sum()

## 06. Regex, Extraction & Numeric Cleaning

In [ ]:
# ===== 06. REGEX, EXTRACTION & NUMERIC CLEANING =====

# Extract integer or decimal room counts (e.g. 3 / 3,5 Zimmer)
df['rooms'] = pd.to_numeric(df['rooms_area_price_raw'].str.extract(r'(\d+(?:[.,]\d+)?)\s*Zimmer')[0].str.replace(',', '.', regex=False), errors='coerce')

# Extract living area (e.g. 49,5 m²)
df['area_m2'] = pd.to_numeric(df['rooms_area_price_raw'].str.extract(r'(\d+(?:[.,]\d+)?)\s*m²')[0].str.replace(',', '.', regex=False), errors='coerce')

# Extract weight in kg (e.g. 12,8 kg)
df['weight_kg'] = pd.to_numeric(df['weight_raw'].str.extract(r'(\d+(?:[.,]\d+)?)\s*kg')[0].str.replace(',', '.', regex=False), errors='coerce')

# Extract percentages (e.g. 12,5 % -> 12.5)
df['percentage'] = pd.to_numeric(df['percentage_raw'].str.extract(r'(\d+(?:[.,]\d+)?)\s*%')[0].str.replace(',', '.', regex=False), errors='coerce')

# Extract car mileage including thousands separators (75'000 km -> 75000)
df['mileage'] = pd.to_numeric(df['mileage_raw'].str.extract(r"([\d'’]+)\s*km")[0].str.replace(r"['’]", '', regex=True), errors='coerce')

# Extract horsepower (245 PS -> 245)
df['hp'] = pd.to_numeric(df['hp_raw'].str.extract(r'(\d+(?:[.,]\d+)?)\s*PS')[0].str.replace(',', '.', regex=False), errors='coerce')

# Extract CHF amounts with apostrophes, decimals and .– endings ★
df['price_chf'] = pd.to_numeric(df['price_raw'].str.extract(r"CHF\s*([\d'’]+(?:[.,]\d{2})?)")[0].str.replace(r"['’]", '', regex=True).str.replace(',', '.', regex=False), errors='coerce')

# Extract integer CHF amounts by joining digit groups (ONLY for prices without cents!) ★
df['price_chf'] = pd.to_numeric(df['price_raw'].str.findall(r'\d+').str.join(''), errors='coerce')

# Extract a numeric substring from text
df['number'] = pd.to_numeric(df['raw'].str.extract(r'(\d+)')[0], errors='coerce')

# Find ALL digit groups in a sample string
re.findall(r'\d+', "CHF 1'249.–")

# Extract all matched text fragments from a column
df['matches'] = df['raw'].str.findall(r'\d+')

# Replace Swiss thousands apostrophes with nothing
df['raw_clean'] = df['raw'].str.replace(r"['’]", '', regex=True)

# Convert text values with decimal commas to numbers
df['number'] = pd.to_numeric(df['raw'].str.replace(',', '.', regex=False), errors='coerce')

## 07. String Cleaning, Lenghts & Keyword Flags

In [ ]:
# ===== 07. STRING CLEANING, LENGTHS & KEYWORD FLAGS =====

# Convert text to lowercase
df['description_raw'] = df['description_raw'].str.lower()

# Convert text to uppercase
df['description_raw'] = df['description_raw'].str.upper()

# Capitalize the first character
df['description_raw'] = df['description_raw'].str.capitalize()

# Remove whitespace from both ends
df['description_raw'] = df['description_raw'].str.strip()

# Replace a non-breaking space (common in HTML text)
df['description_raw'] = df['description_raw'].str.replace('\xa0', ' ', regex=False)

# Remove whitespace on the left only
df['description_raw'] = df['description_raw'].str.lstrip()

# Remove whitespace on the right only
df['description_raw'] = df['description_raw'].str.rstrip()

# Replace specific text literally
df['description_raw'] = df['description_raw'].str.replace('Wohnung', 'Apartment', regex=False)

# Replace multiple whitespace characters with a single space
df['description_raw'] = df['description_raw'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Treat empty or whitespace-only strings as missing values
df['description_raw'] = df['description_raw'].replace(r'^\s*$', pd.NA, regex=True)

# Standardize text to uppercase and trim surrounding whitespace
df['type'] = df['type'].str.upper().str.strip()

# Split text at the first space (e.g. brand and model)
df['type_parts'] = df['type'].str.split(' ', n=1)

# Count digits contained in each string
df['type_digit_count'] = df['type'].str.count(r'\d')

# Calculate text length (missing values remain missing)
df['description_len'] = df['description_raw'].str.len()

# Calculate text length with apply (only if there are no missing strings)
df['description_len'] = df['description_raw'].apply(len)

# Find one of several keywords regardless of uppercase/lowercase
df['match'] = df['description_raw'].str.contains(r'RENOVIERT|SANIERT|NEUBAU', case=False, na=False)

# Create a binary 0/1 keyword flag
df['match'] = df['description_raw'].str.contains(r'RENOVIERT|SANIERT|NEUBAU', case=False, na=False).astype(int)

# Count rows matching a keyword
df['match'].sum()

# Show matching text rows
df.loc[df['match'] == 1, ['description_raw', 'match']]

## 08. New Numeric Variables, Conditions & Dates

In [ ]:
# ===== 08. NEW NUMERIC VARIABLES, CONDITIONS & DATES =====

# Calculate a price-to-horsepower ratio (avoid division by zero)
df['price_per_hp'] = (df['price_chf'] / df['hp'].replace(0, np.nan)).round(2)

# Increase a price by 8.1 percent
df['price_plus_tax'] = (df['price_chf'] * 1.081).round(2)

# Add a fixed shipping charge
df['price_shipping'] = (df['price_chf'] + 10).round(2)

# Subtract a fixed discount
df['price_discount'] = (df['price_chf'] - 50).round(2)

# Create a numeric 0/1 flag and preserve missing prices
df['expensive_flag'] = (df['price_chf'].astype('Float64') > 1000).astype('Int64')

# Create a categorical flag with two labels
df['price_class'] = pd.Series(np.where(df['price_chf'].gt(1000), 'teuer', 'günstig'), index=df.index).where(df['price_chf'].notna())

# Create a combined flag using brand AND price
df['samsung_expensive'] = ((df['brand'] == 'Samsung') & (df['price_chf'].astype('Float64') > 1000)).astype('Int64')

# Store the current datetime as formatted text
df['timestamp_text'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Store the current datetime as a pandas timestamp
df['timestamp'] = pd.Timestamp.now()

# Store the current timestamp in Zurich time
df['timestamp_ch'] = pd.Timestamp.now(tz='Europe/Zurich')

# Convert date strings to dates (invalid values -> NaT)
df['date'] = pd.to_datetime(df['date_raw'], format='%d.%m.%Y', errors='coerce')

# Extract the year from a date
df['year'] = df['date'].dt.year.astype('Int64')

# Extract the month from a date
df['month'] = df['date'].dt.month.astype('Int64')

# Extract the day from a date
df['day'] = df['date'].dt.day.astype('Int64')

# Extract the weekday (Monday=0, Sunday=6)
df['weekday'] = df['date'].dt.dayofweek.astype('Int64')

## 09. Descriptive Statistics, Outliers & EDA

In [ ]:
# ===== 09. DESCRIPTIVE STATISTICS, OUTLIERS & EDA =====

# Calculate the mean
df['price_chf'].mean()

# Calculate the median ★
df['price_chf'].median()

# Find the most frequent values (mode; may return more than one)
df['price_chf'].mode()

# Get all common summary measures at once
df['price_chf'].agg(['count', 'mean', 'median', 'min', 'max', 'var', 'std', 'skew', 'kurt'])

# Calculate the 5th, 25th, 50th, 75th and 95th percentiles
df['price_chf'].quantile([0.05, 0.25, 0.50, 0.75, 0.95]).round(2)

# Calculate quantiles for several numeric columns
df[['price_chf', 'hp']].quantile([0.25, 0.50, 0.75]).round(2)

# Calculate the covariance between two numeric variables
df['price_chf'].cov(df['hp'])

# Calculate the covariance matrix for selected numeric variables
df[['price_chf', 'hp', 'mileage']].cov().round(2)

# Calculate excess kurtosis (tail weight and peakedness)
df['price_chf'].kurtosis()

# Calculate a correlation matrix across selected numeric columns
df[['price_chf', 'hp', 'mileage']].corr(method='pearson').round(2)

# Calculate standard deviation
df['price_chf'].std()

# Calculate variance
df['price_chf'].var()

# Calculate skewness ★
df['price_chf'].skew()

# Calculate the 25th percentile (Q1)
q1 = df['price_chf'].quantile(0.25)

# Calculate the 75th percentile (Q3)
q3 = df['price_chf'].quantile(0.75)

# Calculate the interquartile range (IQR)
iqr = q3 - q1

# Calculate the lower IQR outlier bound
lower = q1 - 1.5 * iqr

# Calculate the upper IQR outlier bound
upper = q3 + 1.5 * iqr

# Mark potential outliers using the 1.5 × IQR rule
df['outlier'] = ((df['price_chf'] < lower) | (df['price_chf'] > upper)).where(df['price_chf'].notna())

# Count marked outliers
df['outlier'].sum()

# Show potential outliers
df.loc[df['outlier'].fillna(False), ['brand', 'price_chf']]

# Make a copy without marked outliers (only if justified)
df_no_outliers = df.loc[~df['outlier'].fillna(False)].copy()

# Exclude a confirmed erroneous record by its product ID (preferred over value)
df_no_error = df.loc[df['product_id'] != 'INCORRECT_ID'].copy()

# Show the most expensive products to investigate recording errors ★
df[['brand', 'model', 'price_chf']].sort_values('price_chf', ascending=False).head(10)

# Find the median price per brand and rank it ★
df.groupby('brand')['price_chf'].median().sort_values(ascending=False)

# Compare mean prices per brand (outlier-sensitive) ★
df.groupby('brand')['price_chf'].mean().sort_values(ascending=False)

## 10. Discretization & Binning

In [ ]:
# ===== 10. DISCRETIZATION / BINNING =====

# Create 3 equal-width bins using cut
df['segment'] = pd.cut(df['price_chf'], bins=3, labels=['günstig', 'mittel', 'teuer'])

# Create 3 approximately equal-sized quantile bins using qcut ★
df['segment'] = pd.qcut(df['price_chf'], q=3, labels=['günstig', 'mittel', 'teuer'])

# Create classes with explicit boundaries (0–500, 500–1000, >1000)
df['segment'] = pd.cut(df['price_chf'], bins=[0, 500, 1000, np.inf], labels=['günstig', 'mittel', 'teuer'], right=True, include_lowest=True)

# Count values in bins including NaN
df['segment'].value_counts(sort=False, dropna=False)

# Learn and assign bins with KBinsDiscretizer (valid non-missing values only)
df.loc[df['price_chf'].notna(), 'price_bin'] = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform').fit_transform(df.loc[df['price_chf'].notna(), ['price_chf']]).ravel().astype(int)

# Build a brand-by-price-segment contingency table ★
pd.crosstab(df['brand'], df['segment'])

# Build a contingency table with margins (totals)
pd.crosstab(df['brand'], df['segment'], margins=True)

# Build a row-normalized contingency table in percent
pd.crosstab(df['brand'], df['segment'], normalize='index').mul(100).round(2)

## 11. One-Hot-Encoding

In [ ]:
# ===== 11. ONE-HOT ENCODING =====

# One-hot encode a categorical column (all 0/1 indicator columns)
df_encoded = pd.get_dummies(df, columns=['brand'], drop_first=False, dtype=int)

# One-hot encode and omit the first category (avoid redundant dummy columns)
df_encoded = pd.get_dummies(df, columns=['brand'], drop_first=True, dtype=int)

# Remove old dummy columns before re-encoding
df = df.loc[:, ~df.columns.str.startswith('brand_')].copy()

# Create a scikit-learn OneHotEncoder that handles unseen categories
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=int)

# Fit and one-hot encode a categorical column (non-missing rows only)
encoded_values = encoder.fit_transform(df.loc[df['brand'].notna(), ['brand']])

# Retrieve one-hot encoded column names
encoded_columns = encoder.get_feature_names_out(['brand'])

# Attach the encoded columns to their original row indices
df = df.join(pd.DataFrame(encoded_values, columns=encoded_columns, index=df.index[df['brand'].notna()]))

## 12. Scaling (Manual + Scikit-Learn)

In [ ]:
# ===== 12. SCALING (MANUAL + SCIKIT-LEARN) =====

# Min-Max scaling by hand (usually to [0, 1])
df['value_minmax'] = (df['value'] - df['value'].min()) / (df['value'].max() - df['value'].min())

# Min-Max scaling using scikit-learn
df['value_minmax_sk'] = MinMaxScaler().fit_transform(df[['value']]).ravel()

# Max-Absolute scaling by hand (usually to [-1, 1])
df['value_maxabs'] = df['value'] / df['value'].abs().max()

# Max-Absolute scaling using scikit-learn
df['value_maxabs_sk'] = MaxAbsScaler().fit_transform(df[['value']]).ravel()

# Robust scaling by hand (median and IQR)
df['value_robust'] = (df['value'] - df['value'].median()) / (df['value'].quantile(0.75) - df['value'].quantile(0.25))

# Robust scaling using scikit-learn
df['value_robust_sk'] = RobustScaler().fit_transform(df[['value']]).ravel()

# Z-score standardization by hand (sample std, ddof=1)
df['value_zscore'] = (df['value'] - df['value'].mean()) / df['value'].std()

# Z-score standardization using scikit-learn (population std, ddof=0)
df['value_zscore_sk'] = StandardScaler().fit_transform(df[['value']]).ravel()

# Match scikit-learn's ddof=0 convention manually
df['value_zscore_ddof0'] = (df['value'] - df['value'].mean()) / df['value'].std(ddof=0)

# Compare manual and scikit-learn scaling results
np.allclose(df['value_zscore_ddof0'], df['value_zscore_sk'], equal_nan=True)

## 13. Data Transformations

In [ ]:
# ===== 13. DATA TRANSFORMATIONS =====

# Apply natural logarithm to strictly positive values (others -> NaN)
df['value_log'] = np.log(df['value'].where(df['value'] > 0))

# Apply log(1+x) to non-negative values (including zero)
df['value_log1p'] = np.log1p(df['value'].where(df['value'] >= 0))

# Apply square-root transformation (x must be >= 0)
df['value_sqrt'] = np.sqrt(df['value'].where(df['value'] >= 0))

# Select valid strictly positive observations for Box-Cox
boxcox_mask = df['value'].gt(0) & df['value'].notna()

# Apply Box-Cox and obtain the chosen lambda (positive nonconstant values)
df.loc[boxcox_mask, 'value_boxcox'], boxcox_lambda = boxcox(df.loc[boxcox_mask, 'value'])

# Apply inverse/reciprocal transformation (zero -> NaN)
df['value_inverse'] = 1 / df['value'].replace(0, np.nan)

# Normalize by the maximum before applying exponential transformation
df['value_ratio'] = df['value'] / df['value'].max()

# Apply exponential transformation to the normalized values
df['value_exp'] = np.exp(df['value_ratio'])

## 14. Combining, Grouping & Reshaping

In [ ]:
# ===== 14. COMBINING, GROUPING & RESHAPING =====

# Merge two DataFrames on a common ID (INNER JOIN: common keys only) ★
df_merged = df.merge(df2, on='product_id', how='inner')

# Merge two DataFrames and keep all left-hand rows
df_merged = df.merge(df2, on='product_id', how='left')

# Merge two DataFrames and keep all rows from both
df_merged = df.merge(df2, on='product_id', how='outer')

# Merge using differently named key columns
df_merged = df.merge(df2, left_on='product_id', right_on='review_id', how='inner')

# Join on a left-side column and the right DataFrame index
df_joined = df.join(df2.set_index('product_id'), on='product_id', how='left', rsuffix='_review')

# Concatenate two DataFrames vertically (add rows)
df_combined = pd.concat([df, df2], axis=0, ignore_index=True)

# Concatenate two DataFrames horizontally (align by index)
df_combined = pd.concat([df, df2], axis=1)

# Get the number of records in each group
df.groupby('brand').size().sort_values(ascending=False)

# Calculate mean values for each group
df.groupby('brand')['price_chf'].mean().round(2)

# Calculate median values for each group ★
df.groupby('brand')['price_chf'].median().round(2)

# Calculate several statistics for multiple numerical variables
df[['price_chf', 'hp']].agg(['mean', 'median', 'std']).round(2)

# Get the most frequent value of a categorical column
df['brand'].mode()

# Count category combinations in a contingency table (include missing categories)
pd.crosstab(df['brand'], df['segment'], dropna=False)

# Calculate count, mean, median and maximum per group
df.groupby('brand').agg(n=('price_chf', 'size'), mean_price=('price_chf', 'mean'), median_price=('price_chf', 'median'), max_price=('price_chf', 'max')).round(2)

# Create a pivot table with mean prices per brand and segment
df_pivot = pd.pivot_table(df, index='brand', columns='segment', values='price_chf', aggfunc='mean')

# Create a pivot table with counts (including missing observations)
df_pivot = pd.pivot_table(df, index='brand', columns='segment', values='product_id', aggfunc='size', fill_value=0, observed=False)

# Create a wide-to-long table with melt
df_long = df.melt(id_vars=['product_id'], value_vars=['price_chf', 'hp'], var_name='variable', value_name='value')

# Create a long-to-wide table with pivot (keys must be unique)
df_wide = df_long.pivot(index='product_id', columns='variable', values='value')

# Stack DataFrame columns into a row-index level
df_stacked = df[['price_chf', 'hp']].stack()

# Unstack the last index level back into columns
df_unstacked = df_stacked.unstack()

## 15. Correlation & Visualization

In [ ]:
# ===== 15. CORRELATION & VISUALIZATION =====

# Calculate Pearson correlation (linear relationship)
df[['price_chf', 'avg_score']].corr(method='pearson').round(2)

# Calculate Spearman rank correlation (monotonic relationship) ★
df[['price_chf', 'avg_score']].corr(method='spearman').round(2)

# Calculate a single Spearman correlation coefficient ★
df['price_chf'].corr(df['avg_score'], method='spearman').round(2)

# Draw a histogram of numerical values
df['price_chf'].hist(bins=25)

# Draw a histogram with a density curve
sns.histplot(data=df, x='price_chf', bins=25, kde=True)

# Draw a boxplot for outlier detection ★
sns.boxplot(data=df, x='price_chf')

# Draw two histograms side-by-side
axes = df[['value', 'value_minmax']].hist(bins=25, layout=(1, 2), figsize=(10, 4))

# Draw a scatterplot with axis labels ★
ax = sns.scatterplot(data=df, x='price_chf', y='avg_score'); ax.set(xlabel='Preis (CHF)', ylabel='Durchschnittliche Bewertung')

# Draw a pandas scatterplot with a title
df.plot.scatter(x='price_chf', y='avg_score', title='Preis vs. Bewertung')

# Draw a boxplot grouped by brand
sns.boxplot(data=df, x='brand', y='price_chf')

# Draw a bar chart from grouped median prices
df.groupby('brand')['price_chf'].median().plot.bar()

# Draw a line chart for a numerical trend (sort x values first)
df.sort_values('date').plot(x='date', y='price_chf', kind='line', marker='o')

# Draw a bar chart of the ten most frequent categories
df['brand'].value_counts().head(10).plot.barh()

# Draw a count plot (category frequencies)
sns.countplot(data=df, x='brand')

# Draw a Q-Q plot to compare sample quantiles with a normal distribution
stats.probplot(df['price_chf'].dropna(), dist='norm', plot=plt)

# Draw a scatterplot matrix of selected numeric variables
sns.pairplot(df[['price_chf', 'hp', 'mileage']].dropna())

# Draw a hexbin plot to show density for overlapping numerical observations
df.plot.hexbin(x='price_chf', y='hp', gridsize=25)

# Draw a Pearson correlation heatmap with coefficients
sns.heatmap(df[['price_chf', 'hp', 'mileage']].corr(method='pearson'), annot=True, fmt='.2f', center=0, vmin=-1, vmax=1)

# Draw a violin plot to compare distributions across categories
sns.violinplot(data=df, x='brand', y='price_chf', inner='quartile')

# Draw a scatterplot with categories indicated by color
sns.scatterplot(data=df, x='price_chf', y='hp', hue='brand')

# Draw a line chart from an aggregated time series
df.groupby('date')['price_chf'].mean().plot.line(marker='o')

# Draw a boxplot directly from a Pandas Series
df['price_chf'].plot.box()

# Set the title and axis labels on the current Matplotlib plot
plt.title('Example plot'); plt.xlabel('X axis'); plt.ylabel('Y axis')

# Rotate crowded tick labels and optimize the figure layout
plt.xticks(rotation=45); plt.tight_layout()

# Create a new Matplotlib figure and axes for further plotting
fig, ax = plt.subplots(figsize=(8, 5))

# Enable a grid and legend where needed
plt.grid(True); plt.legend()

# Show / finalize an existing Matplotlib figure
plt.show()

## 16. HTML / Web Scraping with Beautifulsoup

In [ ]:
# ===== 16. HTML / WEB SCRAPING WITH BEAUTIFULSOUP =====

# Parse local HTML text with BeautifulSoup ★
soup = BeautifulSoup(html_content, 'html.parser')

# Select the SECOND HTML table (index 1) ★
table = soup.find_all('table')[1]

# Select a table by its CSS selector (adapt after inspecting HTML)
table = soup.select_one('table.specifications')

# Find all table rows ★
html_rows = table.find_all('tr')

# Extract TD text from each row (headers without TD become empty lists) ★
rows = [[cell.get_text(strip=True) for cell in row.find_all('td')] for row in html_rows]

# Create a DataFrame with two table columns ★
df_html = pd.DataFrame(rows, columns=['merkmal', 'wert'])

# Remove empty heading rows and reset the index ★
df_html = df_html.dropna().reset_index(drop=True)

# Extract only rows with exactly two TD cells (alternative to dropna)
df_html = pd.DataFrame([r for r in rows if len(r) == 2], columns=['merkmal', 'wert'])

# Extract the text of the first matching HTML element
soup.find('h1').get_text(strip=True)

# Extract all product elements using a CSS selector (adapt to the HTML)
products = soup.select('.product-card')

# Extract an HTML attribute from the first product card
products[0].get('data-product-id')

# Extract product-card text using CSS selectors
products[0].select_one('span.brand').get_text(strip=True).replace('\xa0', ' ')

# Parse multiple locally stored shop pages
page_soups = [BeautifulSoup(Path(f'html/shop_page_{p}.html').read_text(encoding='utf-8'), 'html.parser') for p in [1, 2, 3]]

# Build a list of all product cards from multiple local shop pages
cards = [card for page_soup in page_soups for card in page_soup.select('div.product-card')]

# Extract product IDs and brands into a clean table (adapt selectors and columns)
df_raw = pd.DataFrame([{'product_id': c.get('data-product-id'), 'brand': c.select_one('span.brand').get_text(strip=True)} for c in cards])


## Interpretationen & typische Prüfungsfallen – nur das Nötigste

| Begriff / Methode | Kurzinterpretation |
|---|---|
| **MCAR** | Fehlende Werte sind unabhängig von beobachteten und unbeobachteten Variablen. Vollständige Fälle können weniger effizient sein, führen aber unter MCAR nicht allein wegen des Fehlens zu systematischer Verzerrung |
| **MAR** | Fehlende Werte hängen (bedingt auf beobachtete Variablen) nicht zusätzlich vom fehlenden Wert selbst ab; Gruppenunterschiede können Hinweise geben |
| **MNAR** | Fehlwahrscheinlichkeit hängt auch vom fehlenden Wert ab, z. B. sehr hohe Preise werden absichtlich ausgeblendet. `dropna()` / Mittelwert-Imputation kann hier zu verzerrten Schätzungen führen |
| **Mean vs. Median** | Mittelwert reagiert stärker auf extreme Werte, Median ist robuster. Positive Schiefe und Mittelwert > Median deuten oft auf Rechtsschiefe |
| **Ausreisser** | Erst mit `describe()`, Sortierung und Boxplot prüfen; ein Ausreisser ist nicht automatisch ein Datenfehler. Konkreten Erfassungsfehler fachlich begründet korrigieren/ausschliessen |
| **Index nach dem Drop** | Nach `dropna()`/`drop_duplicates()` kann der Index Lücken haben. Bei Listen-Zuweisungen als `pd.Series(...)` sind **Indexausrichtung** und `reset_index(drop=True)` wichtig |
| **Regex / CHF** | `re.findall(r'\d+', ...)` liefert für `1'249` zwei Treffer. **Nur** den ersten zu nehmen macht `1` statt `1249`; Zusammenfügen aller Ziffern funktioniert **nicht** bei Rappen (`1'249.90` → `124990`). Verwende das CHF-Muster mit Decimal-Unterstützung |
| **`cut()` / `qcut()`** | `cut`: feste oder gleich breite Zahlenintervalle. `qcut`: über Quantile ungefähr gleich viele Beobachtungen je Klasse. Fehlende Werte bleiben `NaN`; gleiche Quantilgrenzen können bei vielen identischen Werten Probleme verursachen |
| **KBinsDiscretizer** | Lernbasierte numerische Klassen mit `uniform` (gleich breit), `quantile` (ähnliche Häufigkeit), `kmeans` (Cluster). Bei fehlenden Werten geeignete Zeilen vorab auswählen |
| **One-Hot Encoding** | Kategorie in 0/1-Spalten; `drop_first=True` vermeidet eine redundante Kategorie. `OneHotEncoder(handle_unknown='ignore')` erlaubt unbekannte Kategorien beim späteren Transformieren. `get_dummies` kann fehlende Kategorien standardmässig als lauter Nullen codieren |
| **Scaling** | Min-Max: 0–1; Max-Abs: etwa −1 bis 1; Robust: Median/IQR (robust gegen Ausreisser); Z-Score: Mittelwert 0, Standardabweichung 1. `StandardScaler` verwendet `ddof=0`, `Series.std()` standardmässig `ddof=1`. Bei konstanten Spalten können manuelle Formeln durch 0 teilen |
| **Transformation** | Log/Box-Cox benötigen strikt positive Werte; `log1p` und SQRT erlauben 0; Inverse benötigt ≠ 0; Exponential kann bei grossen Zahlen überlaufen. Box-Cox benötigt für die gezeigte Variante positive, nichtkonstante Werte |
| **`merge()`** | Default `how='inner'`: nur IDs in beiden Tabellen. `left`: alle linken IDs; nicht eindeutige Join-Keys können die Zeilenzahl vervielfachen |
| **`pivot()` / `pivot_table()`** | `pivot()` verlangt eindeutige Index-Spalten-Kombinationen. `pivot_table()` kann Duplikate via `aggfunc` aggregieren. `melt()` wandelt breit → lang; `stack()` verschiebt Spalten in Indexebene |
| **Korrelation** | Pearson: linearer Zusammenhang; Spearman: monotoner Rangzusammenhang. Ausreisser und nichtlineare Muster im Scatterplot kontrollieren. **Korrelation ≠ Kausalität** (Anscombe-Problem) |
| **HTML** | `find_all('table')[1]` nur nach Prüfung der HTML-Struktur. Zwischenüberschriften ohne `<td>` werden zu `[]` → DataFrame-NaNs → `dropna()` entfernt sie |
